# BLUEFIELD 2 commands

In [ ]:
access:

ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@152.54.15.39	
ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@152.54.15.35	

## Configuring management interface (tmfifo_net0) and PF:

### Run on the host:

In [ ]:
sudo ip addr add 192.168.100.1/30 dev tmfifo_net0

sudo ip addr add 192.168.1.2/24 dev enp7s0np0

sudo ip link set enp7s0np0 up

## Connecting BF to internet

### Run on the host:

In [ ]:
sudo iptables -t nat -A POSTROUTING -o enp3s0 -j MASQUERADE
sudo iptables -A FORWARD -i enp3s0 -o tmfifo_net0 -m state --state RELATED,ESTABLISHED -j ACCEPT
sudo iptables -A FORWARD -i tmfifo_net0 -o enp3s0 -j ACCEPT
echo 1 | sudo tee /proc/sys/net/ipv4/ip_forward

## Add the DNS nameserver in the SmartNIC:

In [ ]:
vim /etc/resolv.conf

## IPV6

## Compiling DOCA apps

### Install meson:

sudo apt update
sudo apt install meson

### Export necessary libs (https://docs.nvidia.com/doca/archive/doca-v2.2.0/troubleshooting/index.html):

In [ ]:
export PKG_CONFIG_PATH=${PKG_CONFIG_PATH}:/opt/mellanox/doca/lib/aarch64-linux-gnu/pkgconfig

export PATH=${PATH}:/opt/mellanox/doca/tools

export PKG_CONFIG_PATH=${PKG_CONFIG_PATH}:/opt/mellanox/dpdk/lib/aarch64-linux-gnu/pkgconfig

export PKG_CONFIG_PATH=${PKG_CONFIG_PATH}:/opt/mellanox/flexio/lib/pkgconfig
export PKG_CONFIG_PATH=${PKG_CONFIG_PATH}:/opt/mellanox/dpdk/include/aarch64-linux-gnu/dpdk

for x86
export PKG_CONFIG_PATH=${PKG_CONFIG_PATH}:/opt/mellanox/dpdk/include/dpdk/
export PKG_CONFIG_PATH=${PKG_CONFIG_PATH}:/opt/mellanox/dpdk/include/x86_64-linux-gnu/dpdk/
export PKG_CONFIG_PATH=${PKG_CONFIG_PATH}:/opt/mellanox/dpdk/lib/x86_64-linux-gnu/
export PKG_CONFIG_PATH=${PKG_CONFIG_PATH}:/opt/mellanox/dpdk/lib/x86_64-linux-gnu/pkgconfig/

Depending on the application, it may need other libs (see meson output and add lib)

### Configuring hugepages:

In [ ]:
echo 2048 > /sys/kernel/mm/hugepages/hugepages-2048kB/nr_hugepages
sudo mkdir /mnt/huge
sudo mount -t hugetlbfs -o pagesize=2M nodev /mnt/huge

sudo sh -c  "echo 1024 > /sys/kernel/mm/hugepages/hugepages-2048kB/nr_hugepages"

### Build a doca flow app

In [ ]:
cd /opt/mellanox/doca/samples/doca_flow/<sample_name>
meson /tmp/build
ninja -C /tmp/build

### Running the app:

In [ ]:
/tmp/build/doca_<sample_name> -a auxiliary:mlx5_core.sf.2 -a auxiliary:mlx5_core.sf.3 -- -l 60

/tmp/build/doca_<sample_name> -a auxiliary:mlx5_core.sf.2,dv_flow_en=2 -a auxiliary:mlx5_core.sf.3,dv_flow_en=2 -- -l 60

/tmp/build/doca_<sample_name> -- -p 03:00.0 -r sf[2-3] -l 60

## Configuring OvS

### hw offload:

ovs-vsctl set Open_vSwitch . other_config:hw-offload=true

### flow aging:

ovs-vsctl set Open_vSwitch . other_config:max-idle=30000

### handler threads:

ovs-vsctl set Open_vSwitch . other_config:n-handler-threads=4
ovs-vsctl set Open_vSwitch . other_config:n-revalidator-threads=4

### basic ovs config:

In [ ]:
# Add topology: Adding physical port of BF, a scalable function and the host representor
ovs-vsctl add-br ovsbr1
ovs-vsctl add-port ovsbr1 p0;
ovs-vsctl add-port ovsbr1 pf0hpf;
ovs-vsctl add-port ovsbr1 en3f0pf0sf2;
ovs-vsctl add-br ovsbr2
ovs-vsctl add-port ovsbr2 p1;
ovs-vsctl add-port ovsbr2 pf1hpf;
ovs-vsctl add-port ovsbr2 en3f0pf0sf3;
ovs-vsctl set Open_vSwitch . other_config:hw-offload=true;
systemctl restart openvswitch-switch
systemctl enable openvswitch-switch

## For most apps:
ovs-vsctl add-br ovsbr1
ovs-vsctl add-port ovsbr1 p0;
ovs-vsctl add-port ovsbr1 en3f0pf0sf2;
ovs-vsctl add-br ovsbr2
ovs-vsctl add-port ovsbr2 pf0hpf;
ovs-vsctl add-port ovsbr2 en3f0pf0sf3;
ovs-vsctl set Open_vSwitch . other_config:hw-offload=true;
systemctl restart openvswitch-switch
systemctl enable openvswitch-switch

# Add flow rules: 
#Rules for allowing traffic from port 0 to the host 
sudo ovs-ofctl add-flow ovsbr1 in_port=p0,actions=output:pf0hpf
sudo ovs-ofctl add-flow ovsbr1 in_port=pf0hpf,actions=output:p0


# Dump flows:
ovs-ofctl dump-flows ovsbr1

# verify hw-offload:
ovs-appctl dpctl/dump-flows -m | grep pf0hpf
ovs-appctl dpctl/dump-flows type=offloaded


In [ ]:
env LD_LIBRARY_PATH=/opt/mellanox/dpdk/lib/aarch64-linux-gnu /opt/mellanox/dpdk/bin/dpdk-testpmd -a 03:00.0,representor=[0,65535] --socket-mem=1024 -- --total-num-mbufs=131000 -i

## Scalable functions

### Show SFs

In [ ]:
mlnx-sf -a show

### Delete SF

In [ ]:
/opt/mellanox/iproute2/sbin/mlxdevm port del <pci/pci_address/index> (example pci/0000:03:00.0/229408)

### Create SF

In [ ]:
mlnx-sf --action create --device 0000:03:00.0 --sfnum 2 --hwaddr 02:25:f2:8d:a2:4c -t

In [ ]:
# Run test pmd
env LD_LIBRARY_PATH=/opt/mellanox/dpdk/lib/aarch64-linux-gnu /opt/mellanox/dpdk/bin/dpdk-testpmd -a 03:00.0,representor=[0,65535] --socket-mem=1024 -- --total-num-mbufs=131000 --burst=64 -i

In [ ]:
git clone https://github.com/mesonbuild/meson.git
tar czf meson.tar.gz meson/
scp meson.tar.gz  ubuntu@192.168.100.2:~
    
#for copying ninja
wget http://ports.ubuntu.com/pool/universe/n/ninja-build/ninja-build_1.10.1-1_arm64.deb
scp ninja-build_1.10.1-1_arm64.deb ubuntu@192.168.100.2:/tmp/

# DoH 

In [4]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager()

slice = fablib.get_slice(name="DoH-BF2")

server1 = slice.get_node(name="Node1")
server2 = slice.get_node(name="Node2")

In [14]:
server1.upload_file('../DoH/rf_model.json','/home/ubuntu/rf_model.json')

<SFTPAttributes: [ size=641369 uid=1000 gid=1000 mode=0o100664 atime=1748689182 mtime=1748689182 ]>

In [15]:
server1.upload_file('../DoH/mlp_model.h','/home/ubuntu/mlp_model.h')

<SFTPAttributes: [ size=6592 uid=1000 gid=1000 mode=0o100664 atime=1748689182 mtime=1748689182 ]>

In [15]:
server1.upload_file('../DoH/mlp_model.h','/home/ubuntu/mlp_model.h')

<SFTPAttributes: [ size=6592 uid=1000 gid=1000 mode=0o100664 atime=1748689182 mtime=1748689182 ]>

In [9]:
server2.download_file('../DoH/dns2tcp1_incremental_features.csv','/home/ubuntu/scripts/dns2tcp1_incremental_features.csv')

In [16]:
server2.download_file('../DoH/chromeadguard_incremental_features.csv','/home/ubuntu/scripts/chromeadguard_incremental_features.csv')

In [11]:
server2.download_file('../DoH/dns2cat1_incremental_features.csv','/home/ubuntu/scripts/dns2cat1_incremental_features.csv')

In [12]:
server2.download_file('../DoH/dns2iodine1_incremental_features.csv','/home/ubuntu/scripts/dns2iodine1_incremental_features.csv')

## Run the doh application on the BF2

In [ ]:
./build/doh_app -a 0000:03:00.0 -a auxiliary:mlx5_core.sf.2 -a auxiliary:mlx5_core.sf.3 -l 0-1

### Rewrite packet hw address and replay pcap

In [ ]:
tcprewrite --enet-smac=10:70:fd:b3:48:56 --enet-dmac=58:a2:e1:17:30:52 --infile=BenignTraffic3.pcap  --outfile=output_benign.pcap
tcpreplay --intf1=enp7s0  output2.pcap

In [ ]:
scp mlp_model.h ubuntu@192.168.100.2:/tmp/
ssh ubuntu@192.168.100.2 "sudo mv /tmp/mlp_model.h /home/ubuntu/doh/"